<a href="https://colab.research.google.com/github/bishalkshah70-art/AI-and-Ml-/blob/main/prompt_enginerring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import tensorflow as tf
import numpy  as np
import matplotlib_inline as ml
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import json as js
import errno as er
import re


In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install youtube-transcript-api

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available()else "cpu"
print(f"[INFO]Using device:{DEVICE}")

In [ ]:
# Helper function
def extract_video_id(youtube_url: str) -> str:
    """
    Extract the 11-character YouTube video ID.
    """
    regex = r"(?:v=)([0-9A-Za-z_-]{11})"
    match = re.search(regex, youtube_url)

    if match:
        return match.group(1)
    else:
        raise ValueError("Invalid YouTube URL provided.")


# Fetch transcript
def fetch_youtube_transcript(video_url: str) -> str:
    """
    Fetches subtitles and combines them into one string.
    """
    video_id = extract_video_id(video_url)

    print(f"[INFO] Extracting transcript for Video ID: {video_id}")

    try:
        transcript = YouTubeTranscriptApi.get_transcript(
            video_id,
            languages=["en"]
        )

        full_transcript = " ".join(
            item["text"] for item in transcript
        )

        print(f"[SUCCESS] Transcript fetched ({len(full_transcript.split())} words)")
        return full_transcript

    except Exception as e:
        raise RuntimeError(f"Failed to retrive transcript:{str(e)}")

In [ ]:
# 2. Load LLM and Tokenizer for summarization
# bart-large-icon is purpose-built for summarization and stays much closer

# to the source text that flat-t5-base, which is generalist instruction model.

MODEL_NAME = "facebook/bart-large-cnn"

print(f"\n[INFO]Loading open-source LLM:{MODEL_NAME}...")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



model =AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(DEVICE)

print(f"[INFO]Model and Tokenizer loaded sucessfully on {DEVICE}.")

In [ ]:
def chunk_text(text: str, tokenizer, max_tokens: int = 900):
    """
    Splits the full transcript into token-bounded chunks so the whole video
    gets summarized instead of just the first ~1500 characters.
    """

    words = text.split()
    chunks = []
    current_chunk_words = []

    for word in words:
        current_chunk_words.append(word)

        candidate = " ".join(current_chunk_words)
        token_len = len(
            tokenizer(candidate, add_special_tokens=False).input_ids
        )

        if token_len >= max_tokens:
            # Remove the last word so we stay under the limit
            current_chunk_words.pop()

            chunks.append(" ".join(current_chunk_words))

            current_chunk_words = [word]

    # Add the final chunk
    if current_chunk_words:
        chunks.append(" ".join(current_chunk_words))

    return chunks


In [ ]:
def summarize_chunk(text: str) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024  # BART's maximum input length
    ).to(DEVICE)

    output_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=150,
        min_new_tokens=40,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )


In [ ]:
def generate_engineered_summary(transcript_text: str) -> str:
    """
    Summarizes the FULL transcript by chunking it, summarizing each chunk,
    then (if there were multiple chunks) summarizing the summaries into
    one coherent final result.
    """

    chunks = chunk_text(transcript_text, tokenizer)

    print(f"[INFO] Transcript split into {len(chunks)} chunk(s) for summarization.")

    chunk_summaries = []

    for i, chunk in enumerate(chunks, start=1):
        print(f"[INFO] Summarizing chunk {i}/{len(chunks)}...")
        chunk_summaries.append(summarize_chunk(chunk))

    if len(chunk_summaries) == 1:
        return chunk_summaries[0]

    # Combine partial summaries and do one final summarization pass
    combined = " ".join(chunk_summaries)

    print("[INFO] Producing final consolidated summary...")

    return summarize_chunk(combined)

In [ ]:
if __name__ == "__main__":
    sample_youtube_url = "https://youtu.be/y6nn4p5JOBM?si=tL8Kx88abOH8oBul"

    try:
        raw_transcript = fetch_youtube_transcript(sample_youtube_url)

        structured_summary = generate_engineered_summary(raw_transcript)

        print("\n" + "=" * 70)
        print("YOUTUBE VIDEO SUMMARIZER — OUTPUT REPORT")
        print("=" * 70)
        print(f"URL: {sample_youtube_url}\n")
        print(f"GENERATED SUMMARY:\n{structured_summary}")
        print("=" * 70)

    except Exception as error:
        print(f"\n[ERROR] Process failed: {error}")